# 🧪 06 - Describing Data Lab: Palmer Penguins

A research team has measured 344 penguins on three islands in Antarctica and sent you `penguins.csv`. They want a **numeric description** of the data before anyone draws a single chart:

- What's in the file, and what does each column mean?
- What's typical for each measurement, and how much does it vary?
- Are there outliers, and are they mistakes?
- How do the three species differ?
- Which measurements are related, and does that hold up within each species?

You'll answer with everything from notebooks 01–05, and save a small report. In **Module 08's lab**, you'll turn these same findings into figures, so keep your answers. You'll check them against the charts.

## ✅ Learning Goals
- Run a full descriptive workflow, from raw file to saved summary tables
- Choose statistics that match each column's type and shape
- Separate real outliers from errors, overall and within groups
- Compare groups fairly, and test whether a relationship survives splitting by group

```text
Load → Inspect → Clean (Module 06) → Data dictionary (01)
     → Center & spread (02) → Shape & outliers (03)
     → Compare groups (04) → Relationships (05) → Export the report
```

## 0. Setup
Everything this lab saves goes into **`06-OutputFiles`**. Upload that folder along with this notebook.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path('06-OutputFiles')
OUT.mkdir(exist_ok=True)

MEASUREMENTS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

## 1. Load and Look (Change Nothing Yet)

In [2]:
penguins = pd.read_csv('penguins.csv')
print(penguins.shape)
penguins.head()

(344, 7)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [3]:
penguins.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


In [4]:
penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

**One row = one penguin.** Four measurements are each missing twice, and `sex` is missing 11 times. Are the measurement gaps in the same two rows? 

In [5]:
penguins[penguins[MEASUREMENTS].isna().any(axis=1)]

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN


Yes: two penguins with **no measurements at all**. They can't contribute anything to a description of measurements.

## 2. Clean (Module 06)
1. Drop the two rows with no measurements.
2. Keep penguins with unknown `sex`: their measurements are still useful. Just remember they'll drop out of any by-sex comparison.
3. Tidy `sex` from `MALE`/`FEMALE` to `Male`/`Female`.
4. Record meaning with dtypes (notebook 01): `species`, `island`, and `sex` are nominal → `category`.

In [6]:
clean = penguins.dropna(subset=MEASUREMENTS).copy()
clean['sex'] = clean['sex'].str.title()

for col in ['species', 'island', 'sex']:
    clean[col] = clean[col].astype('category')

print('Rows before:', len(penguins), '| after:', len(clean))
print(clean['sex'].value_counts(dropna=False))
clean.dtypes

Rows before: 344 | after: 342
sex
Male      168
Female    165
NaN         9
Name: count, dtype: int64


species              category
island               category
bill_length_mm        float64
bill_depth_mm         float64
flipper_length_mm     float64
body_mass_g           float64
sex                  category
dtype: object

## 3. Data Dictionary (Notebook 01)

In [7]:
data_dictionary = pd.DataFrame({
    'column': clean.columns,
    'meaning': ['Penguin species', 'Island where measured', 'Bill (beak) length',
                'Bill (beak) depth', 'Flipper length', 'Body mass', 'Sex'],
    'kind': ['qualitative', 'qualitative', 'quantitative', 'quantitative',
             'quantitative', 'quantitative', 'qualitative'],
    'scale': ['nominal', 'nominal', 'ratio', 'ratio', 'ratio', 'ratio', 'nominal'],
    'dtype': clean.dtypes.astype(str).values,
    'missing': clean.isna().sum().values,
    'distinct': clean.nunique().values,
})
data_dictionary.to_csv(OUT / 'penguins_data_dictionary.csv', index=False)
data_dictionary

,column,meaning,kind,scale,dtype,missing,distinct
0,species,Penguin species,qualitative,nominal,category,0,3
1,island,Island where measured,qualitative,nominal,category,0,3
2,bill_length_mm,Bill (beak) length,quantitative,ratio,float64,0,164
3,bill_depth_mm,Bill (beak) depth,quantitative,ratio,float64,0,80
4,flipper_length_mm,Flipper length,quantitative,ratio,float64,0,55
5,body_mass_g,Body mass,quantitative,ratio,float64,0,94
6,sex,Sex,qualitative,nominal,category,9,2


## 4. Center and Spread (Notebook 02)
One table with every statistic from notebooks 02 and 03, one row per measurement:

In [8]:
overall = clean[MEASUREMENTS].describe().T
overall['IQR'] = overall['75%'] - overall['25%']
overall['skew'] = clean[MEASUREMENTS].skew()
overall.round(2)

,count,mean,std,min,25%,50%,75%,max,IQR,skew
bill_length_mm,342.0,43.92,5.46,32.1,39.22,44.45,48.5,59.6,9.27,0.05
bill_depth_mm,342.0,17.15,1.97,13.1,15.60,17.30,18.7,21.5,3.10,-0.14
flipper_length_mm,342.0,200.92,14.06,172.0,190.00,197.00,213.0,231.0,23.00,0.35
body_mass_g,342.0,4201.75,801.95,2700.0,3550.00,4050.00,4750.0,6300.0,1200.00,0.47


All four measurements have a mean close to their median, and |skew| below 0.5: no strong lopsidedness like Titanic's fares. Mean and std are reasonable summaries here.

But "roughly symmetric overall" can hide something. This file mixes **three species** of very different sizes. Keep that in mind for section 6.

## 5. Shape and Outliers (Notebook 03)
Reuse the fence function from notebook 03, and count outliers in each measurement:

In [9]:
def iqr_fences(series, k=1.5):
    """Return the (lower, upper) outlier fences for a numeric Series."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


for col in MEASUREMENTS:
    low, high = iqr_fences(clean[col])
    n_out = ((clean[col] < low) | (clean[col] > high)).sum()
    print(f'{col:<18} fences {low:8.1f} to {high:8.1f}   outliers: {n_out}')

bill_length_mm     fences     25.3 to     62.4   outliers: 0
bill_depth_mm      fences     10.9 to     23.3   outliers: 0
flipper_length_mm  fences    155.5 to    247.5   outliers: 0
body_mass_g        fences   1750.0 to   6550.0   outliers: 0


No outliers at all across the whole dataset. Now ask notebook 03's Practice 3 question: is a value unusual **for its own species**? Compute the fences within each species with `groupby().transform()`, which lines each group's statistic up with the original rows:

In [10]:
def outliers_within(df, col, group):
    """Boolean mask: is each value outside its own group's IQR fences?"""
    q1 = df.groupby(group, observed=True)[col].transform(lambda s: s.quantile(0.25))
    q3 = df.groupby(group, observed=True)[col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    return (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)


within = pd.DataFrame({col: outliers_within(clean, col, 'species') for col in MEASUREMENTS})
within.sum()

bill_length_mm       1
bill_depth_mm        1
flipper_length_mm    2
body_mass_g          2
dtype: int64

In [11]:
clean[within.any(axis=1)].sort_values('species')

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
19,Adelie,Torgersen,46.0,21.5,194.0,4200.0,Male
28,Adelie,Biscoe,37.9,18.6,172.0,3150.0,Female
129,Adelie,Torgersen,44.1,18.0,210.0,4000.0,Male
189,Chinstrap,Dream,52.0,20.7,210.0,4800.0,Male
190,Chinstrap,Dream,46.9,16.6,192.0,2700.0,Female
253,Gentoo,Biscoe,59.6,17.0,230.0,6050.0,Male


A few penguins are unusual **for their species**, even though nothing is unusual overall. Look at the rows: are any values impossible? (A small Chinstrap is still a possible penguin. A 30 g penguin, or a 3-meter flipper, would not be.) These look like real, unusual animals, so we **keep** them.

> 🧠 Whether a value is an outlier depends on **which group you compare it to**.

## 6. Comparing Species (Notebook 04)

In [12]:
species_summary = (
    clean
    .groupby('species', observed=True)
    .agg(
        penguins=('body_mass_g', 'size'),
        mass_mean=('body_mass_g', 'mean'),
        mass_median=('body_mass_g', 'median'),
        mass_std=('body_mass_g', 'std'),
        flipper_mean=('flipper_length_mm', 'mean'),
        bill_length_mean=('bill_length_mm', 'mean'),
        bill_depth_mean=('bill_depth_mm', 'mean'),
    )
    .round(1)
)
species_summary.to_csv(OUT / 'species_summary.csv')
species_summary

,penguins,mass_mean,mass_median,mass_std,flipper_mean,bill_length_mean,bill_depth_mean
species,,,,,,,
Adelie,151,3700.7,3700.0,458.6,190.0,38.8,18.3
Chinstrap,68,3733.1,3700.0,384.3,195.8,48.8,18.4
Gentoo,123,5076.0,5000.0,504.1,217.2,47.5,15.0


Gentoo penguins are much heavier, with longer flippers, but **shallower** bills. Hold on to that "but". It matters in section 7.

### Where Each Species Lives

In [13]:
pd.crosstab(clean['species'], clean['island'], margins=True)

island,Biscoe,Dream,Torgersen,All
species,,,,
Adelie,44,56,51,151
Chinstrap,0,68,0,68
Gentoo,123,0,0,123
All,167,124,51,342


Chinstraps were measured only on Dream, and Gentoos only on Biscoe. So "Biscoe penguins are heavy" would really be a statement about **Gentoos**: island and species are tangled together (a confounder, notebook 04). Compare islands only **within** a species that lives on more than one (Adelie).

### Species × Sex

In [14]:
mass_by_species_sex = clean.pivot_table(
    values='body_mass_g', index='species', columns='sex', aggfunc='mean', observed=True,
).round(0)
mass_by_species_sex['difference'] = mass_by_species_sex['Male'] - mass_by_species_sex['Female']
mass_by_species_sex

sex,Female,Male,difference
species,,,
Adelie,3369.0,4043.0,674.0
Chinstrap,3527.0,3939.0,412.0
Gentoo,4680.0,5485.0,805.0


Within every species, males are heavier. The penguins with unknown sex aren't in this table (section 2).

## 7. Relationships (Notebook 05)

In [ ]:
corr = clean[MEASUREMENTS].corr()
corr.round(2)


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
bill_length_mm,1.00,-0.24,0.66,0.60
bill_depth_mm,-0.24,1.00,-0.58,-0.47
flipper_length_mm,0.66,-0.58,1.00,0.87
body_mass_g,0.60,-0.47,0.87,1.00


In [16]:
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
upper.stack().dropna().rename('r').sort_values(key=abs, ascending=False).round(3)

flipper_length_mm  body_mass_g          0.871
bill_length_mm     flipper_length_mm    0.656
                   body_mass_g          0.595
bill_depth_mm      flipper_length_mm   -0.584
                   body_mass_g         -0.472
bill_length_mm     bill_depth_mm       -0.235
Name: r, dtype: float64

Flipper length and body mass are very strongly related (bigger penguins, bigger flippers). Now look at **bill length vs. bill depth**: a weak **negative** correlation. Longer bills are shallower?

Check whether that holds up **within each species**:

In [17]:
rows = []
for species, group in clean.groupby('species', observed=True):
    rows.append({
        'species': species,
        'n': len(group),
        'r_bill_length_depth': group['bill_length_mm'].corr(group['bill_depth_mm']),
    })
rows.append({'species': 'ALL (combined)', 'n': len(clean),
             'r_bill_length_depth': clean['bill_length_mm'].corr(clean['bill_depth_mm'])})

bill_corr = pd.DataFrame(rows).round(3)
bill_corr.to_csv(OUT / 'bill_correlation_by_species.csv', index=False)
bill_corr

,species,n,r_bill_length_depth
0,Adelie,151,0.391
1,Chinstrap,68,0.654
2,Gentoo,123,0.643
3,ALL (combined),342,-0.235


🤯 **Within every species, the correlation is positive, and moderate to strong.** Combined, it's negative.

This is **Simpson's paradox**: a relationship that reverses when you combine groups. The species summary in section 6 explains how. Gentoos have long bills **and** shallow bills compared with the others. Pooling a "long and shallow" group with "shorter and deeper" groups creates a downward trend that doesn't exist inside any one species. `species` is the confounder.

You found this with numbers alone, but you had to know to split by species. In Module 08's lab, one scatter plot colored by species shows it instantly, with no need to know ahead of time.

## 8. Export the Report

In [18]:
clean.to_csv(OUT / 'cleaned_penguins.csv', index=False)
overall.round(2).to_csv(OUT / 'measurement_summary.csv')
corr.round(3).to_csv(OUT / 'correlation_matrix.csv')

print('Saved:', sorted(p.name for p in OUT.glob('*.csv')))

Saved: ['bill_correlation_by_species.csv', 'cleaned_penguins.csv', 'correlation_matrix.csv', 'measurement_summary.csv', 'penguins_data_dictionary.csv', 'species_summary.csv']


1. Drop the two rows with no measurements.
2. Keep penguins with unknown `sex`: their measurements are still useful. Just remember they'll drop out of any by-sex comparison.
3. Tidy `sex` from `MALE`/`FEMALE` to `Male`/`Female`.
4. Record meaning with dtypes (notebook 01): `species`, `island`, and `sex` are nominal → `category`.

## ❓ Lab Questions
Answer each in the Markdown cell below, with numbers from your output:
1. What does one row represent? How many rows did you remove, and why? Why did you **keep** penguins with unknown sex?
**One row = one penguin.** Four measurements are each missing twice, and `sex` is missing 11 times. Yes: two penguins with **no measurements at all**. They can't contribute anything to a description of measurements so they will be dropped. Keeping penguins with unknown `sex` as their measurements are still useful and they'll drop out of any by-sex comparison.
2. Which measurement varies the most **relative to its mean**? (Hint: divide `std` by `mean`.)
3. Why were there no outliers overall, but some within species? Did you keep or remove them, and why?
4. Which species is heaviest, and by how much (mean and median) compared with the lightest?
5. Why can't you compare islands directly? Which comparison would be fair?
6. Explain the bill length vs. bill depth result in your own words, as if to someone who has never heard of Simpson's paradox.

*Your answers here.*

## 🏋️ Practice

### Practice 1 — Species Spread (easy)
For each species, compute the IQR of `flipper_length_mm`. Which species is the most variable, and is it also the one with the largest std?

In [19]:
penguins = pd.read_csv('penguins.csv')
print(penguins.shape)
penguins.head()

(344, 7)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [23]:
species_stats = (
    penguins.groupby('species')['flipper_length_mm']
    .agg(
        n='count',
        std='std',
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
    )
)

species_stats['iqr'] = species_stats['q3'] - species_stats['q1']

print(
    species_stats[['n', 'iqr', 'std']]
    .sort_values('iqr', ascending=False)
    .round(2)
)

print("Largest IQR:",
      species_stats['iqr'].idxmax())

print("Largest std:",
      species_stats['std'].idxmax())

             n   iqr   std
species                   
Chinstrap   68  10.0  7.13
Adelie     151   9.0  6.54
Gentoo     123   9.0  6.48
Largest IQR: Chinstrap
Largest std: Chinstrap


### Practice 2 — A Fair Island Comparison (medium)
Filter to **Adelie** penguins only (the one species on all three islands). Compare their body mass across islands with size, mean, median, and std. Is there a real island difference among Adelies?

In [26]:
# Your code here
adelie = penguins[penguins['species'] == 'Adelie']
adelie_island=(adelie.groupby('island')['body_mass_g']
               .agg(
                   n='count',
                   mean='mean',
                   median='median',
                   std='std'
               ).round(1).sort_values('mean', ascending=False))
print(adelie_island)

            n    mean  median    std
island                              
Biscoe     44  3709.7  3750.0  487.7
Torgersen  51  3706.4  3700.0  445.1
Dream      56  3688.4  3575.0  455.1


In [27]:
print(
    adelie.groupby('island')['body_mass_g']
    .describe()[['count', 'mean', '50%', 'std']]
    .round(1)
)

           count    mean     50%    std
island                                 
Biscoe      44.0  3709.7  3750.0  487.7
Dream       56.0  3688.4  3575.0  455.1
Torgersen   51.0  3706.4  3700.0  445.1


### Practice 3 — Hunt for Another Reversal (harder)
Loop over **every pair** of measurements. For each pair, compute the combined correlation and the three within-species correlations. Flag any pair where the combined sign differs from **all three** species' signs. Is bill length/depth the only one?

In [28]:
# Your code here

rows = []

for i in range(len(MEASUREMENTS)):
    for j in range(i + 1, len(MEASUREMENTS)):

        x = MEASUREMENTS[i]
        y = MEASUREMENTS[j]

        overall = clean[x].corr(clean[y])

        by_species = (
            clean.groupby('species')
            .apply(lambda g: g[x].corr(g[y]))
        )

        rows.append({
            'pair': f'{x} / {y}',
            'overall': overall,
            'Adelie': by_species['Adelie'],
            'Chinstrap': by_species['Chinstrap'],
            'Gentoo': by_species['Gentoo']
        })

results = pd.DataFrame(rows)

results['flag'] = (
    (np.sign(results['overall']) != np.sign(results['Adelie'])) &
    (np.sign(results['overall']) != np.sign(results['Chinstrap'])) &
    (np.sign(results['overall']) != np.sign(results['Gentoo']))
)

results.round(3)

,pair,overall,Adelie,Chinstrap,Gentoo,flag
0,bill_length_mm / bill_depth_mm,-0.235,0.391,0.654,0.643,True
1,bill_length_mm / flipper_length_mm,0.656,0.326,0.472,0.661,False
2,bill_length_mm / body_mass_g,0.595,0.549,0.514,0.669,False
3,bill_depth_mm / flipper_length_mm,-0.584,0.308,0.580,0.707,True
4,bill_depth_mm / body_mass_g,-0.472,0.576,0.604,0.719,True
5,flipper_length_mm / body_mass_g,0.871,0.468,0.642,0.703,False


In [29]:
results[results['flag']]

,pair,overall,Adelie,Chinstrap,Gentoo,flag
0,bill_length_mm / bill_depth_mm,-0.235053,0.391492,0.653536,0.643384,True
3,bill_depth_mm / flipper_length_mm,-0.583851,0.307620,0.580143,0.706563,True
4,bill_depth_mm / body_mass_g,-0.471916,0.576138,0.604498,0.719085,True


Bill length/depth, Bill depth vs. flipper length and bill depth vs. body mass also have negative overall correlations but positive correlations within each species. This indicates that species membership is acting as a confounder and can reverse the apparent direction of relationships when the species are combined.

## ✏️ Challenge Task
Build a **sex comparison report**: for each species, the number of males and females, and the male − female difference in mean `body_mass_g`, `flipper_length_mm`, and `bill_depth_mm`. Save it as `OUT / 'sex_differences_by_species.csv'`.

In [30]:
# Your code here
sex_stats = (
    clean.groupby(['species', 'sex'], observed=True)
    .agg(
        n=('body_mass_g', 'size'),
        body_mass_g=('body_mass_g', 'mean'),
        flipper_length_mm=('flipper_length_mm', 'mean'),
        bill_depth_mm=('bill_depth_mm', 'mean')
    )
)

wide = sex_stats.unstack('sex')

sex_report = pd.DataFrame({
    'n_female': wide[('n', 'Female')],
    'n_male': wide[('n', 'Male')],
    'body_mass_diff': (
        wide[('body_mass_g', 'Male')]
        - wide[('body_mass_g', 'Female')]
    ),
    'flipper_length_diff': (
        wide[('flipper_length_mm', 'Male')]
        - wide[('flipper_length_mm', 'Female')]
    ),
    'bill_depth_diff': (
        wide[('bill_depth_mm', 'Male')]
        - wide[('bill_depth_mm', 'Female')]
    )
}).round(1)

sex_report.to_csv(
    OUT / 'sex_differences_by_species.csv'
)

sex_report

,n_female,n_male,body_mass_diff,flipper_length_diff,bill_depth_diff
species,,,,,
Adelie,73,73,674.7,4.6,1.5
Chinstrap,34,34,411.8,8.2,1.7
Gentoo,58,61,805.1,8.8,1.5


## 🔥 Challenge (Optional) — Same Workflow, New Data
Run sections 1–7 on `pokemon.csv` in a compact form: data dictionary, measurement summary with skew, outliers overall and within `Legendary`, a group comparison by `Generation`, and the strongest correlations. Save at least one table to `OUT` with a `pokemon_` prefix. Write three findings in a Markdown cell.

In [34]:
# Your code here
poke = pd.read_csv('pokemon.csv')
poke.shape
poke.head()


,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


In [35]:
poke.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   #           800 non-null    int64
 1   Name        800 non-null    str  
 2   Type 1      800 non-null    str  
 3   Type 2      414 non-null    str  
 4   Total       800 non-null    int64
 5   HP          800 non-null    int64
 6   Attack      800 non-null    int64
 7   Defense     800 non-null    int64
 8   Sp. Atk     800 non-null    int64
 9   Sp. Def     800 non-null    int64
 10  Speed       800 non-null    int64
 11  Generation  800 non-null    int64
 12  Legendary   800 non-null    bool 
dtypes: bool(1), int64(9), str(3)
memory usage: 75.9 KB


In [37]:
poke.describe()

,#,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
count,800.000000,800.00000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.00000
mean,362.813750,435.10250,69.258750,79.001250,73.842500,72.820000,71.902500,68.277500,3.32375
std,208.343798,119.96304,25.534669,32.457366,31.183501,32.722294,27.828916,29.060474,1.66129
min,1.000000,180.00000,1.000000,5.000000,5.000000,10.000000,20.000000,5.000000,1.00000
25%,184.750000,330.00000,50.000000,55.000000,50.000000,49.750000,50.000000,45.000000,2.00000
50%,364.500000,450.00000,65.000000,75.000000,70.000000,65.000000,70.000000,65.000000,3.00000
75%,539.250000,515.00000,80.000000,100.000000,90.000000,95.000000,90.000000,90.000000,5.00000
max,721.000000,780.00000,255.000000,190.000000,230.000000,194.000000,230.000000,180.000000,6.00000


In [38]:
poke.isna().sum()

#               0
Name            0
Type 1          0
Type 2        386
Total           0
HP              0
Attack          0
Defense         0
Sp. Atk         0
Sp. Def         0
Speed           0
Generation      0
Legendary       0
dtype: int64

In [ ]:

#Data dictionary/structure
poke.info()

pd.DataFrame({
    'dtype': poke.dtypes,
    'missing': poke.isna().sum(),
    'n_unique': poke.nunique()
})

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   #           800 non-null    int64
 1   Name        800 non-null    str  
 2   Type 1      800 non-null    str  
 3   Type 2      414 non-null    str  
 4   Total       800 non-null    int64
 5   HP          800 non-null    int64
 6   Attack      800 non-null    int64
 7   Defense     800 non-null    int64
 8   Sp. Atk     800 non-null    int64
 9   Sp. Def     800 non-null    int64
 10  Speed       800 non-null    int64
 11  Generation  800 non-null    int64
 12  Legendary   800 non-null    bool 
dtypes: bool(1), int64(9), str(3)
memory usage: 75.9 KB


,dtype,missing,n_unique
#,int64,0,721
Name,str,0,800
Type 1,str,0,18
Type 2,str,386,18
Total,int64,0,200
HP,int64,0,94
Attack,int64,0,111
Defense,int64,0,103
Sp. Atk,int64,0,105
Sp. Def,int64,0,92


In [46]:
# Clean-Up
clean = poke.copy()

print('Rows before:', len(poke), '| after:', len(clean))
clean.dtypes

Rows before: 800 | after: 800


#             int64
Name            str
Type 1          str
Type 2          str
Total         int64
HP            int64
Attack        int64
Defense       int64
Sp. Atk       int64
Sp. Def       int64
Speed         int64
Generation    int64
Legendary      bool
dtype: object

In [47]:
clean = poke.dropna().copy()

In [50]:
# Section 3: Measurement Summary with Skew
MEASUREMENTS = [
    'HP', 'Attack', 'Defense',
    'Sp. Atk', 'Sp. Def', 'Speed',
    'Total'
]

summary = clean[MEASUREMENTS].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max', 'skew']
).T.round(2)

summary


,count,mean,median,std,min,max,skew
HP,414.0,70.65,70.0,22.90,1.0,150.0,0.48
Attack,414.0,83.17,80.0,33.70,10.0,190.0,0.51
Defense,414.0,79.68,75.5,32.59,15.0,230.0,0.99
Sp. Atk,414.0,77.05,70.0,33.81,10.0,180.0,0.65
Sp. Def,414.0,75.57,75.0,27.95,20.0,230.0,0.77
Speed,414.0,70.51,70.0,28.52,5.0,160.0,0.21
Total,414.0,456.63,475.0,120.85,190.0,780.0,0.05


In [51]:
#Saving Summary
summary.to_csv(
    OUT / 'pokemon_measurement_summary.csv'
)

In [52]:
# Section 4: Outliers Overall
for col in MEASUREMENTS:
    low, high = iqr_fences(clean[col])

    n_out = (
        (clean[col] < low) |
        (clean[col] > high)
    ).sum()

    print(
        f'{col:<10}  outliers: {n_out}'
    )

HP          outliers: 5
Attack      outliers: 6
Defense     outliers: 7
Sp. Atk     outliers: 1
Sp. Def     outliers: 1
Speed       outliers: 1
Total       outliers: 0


In [54]:
# Section 5: Outliers Within Legendary Status
for legendary, group in clean.groupby('Legendary'):

    print(f'\nLegendary = {legendary}')

    for col in MEASUREMENTS:
        low, high = iqr_fences(group[col])

        n_out = (
            (group[col] < low) |
            (group[col] > high)
        ).sum()

        print(f'{col:<10} {n_out}')


Legendary = False
HP         6
Attack     5
Defense    8
Sp. Atk    4
Sp. Def    4
Speed      2
Total      0

Legendary = True
HP         2
Attack     3
Defense    2
Sp. Atk    0
Sp. Def    0
Speed      1
Total      0


In [55]:
# Section 6: Group Comparison by Generation
generation_summary = (
    clean.groupby('Generation')
    .agg(
        n=('Total', 'size'),
        mean_total=('Total', 'mean'),
        median_total=('Total', 'median'),
        mean_attack=('Attack', 'mean'),
        legendary_rate=('Legendary', 'mean')
    )
    .round(2)
)

generation_summary['legendary_rate'] *= 100

generation_summary

,n,mean_total,median_total,mean_attack,legendary_rate
Generation,,,,,
1,78,447.36,462.5,81.21,5.0
2,55,436.62,430.0,75.04,4.0
3,82,453.99,460.0,86.05,10.0
4,67,478.75,500.0,85.13,9.0
5,82,462.15,481.5,86.05,16.0
6,50,458.74,494.0,83.14,14.0


In [58]:
# saving summary
generation_summary.to_csv(
    OUT / 'pokemon_generation_summary.csv'
)

In [60]:
# Section 7: Strongest Correlations
corr = clean[MEASUREMENTS].corr()

upper = corr.where(
    np.triu(
        np.ones(corr.shape, dtype=bool),
        k=1
    )
)

pairs = (
    upper.stack()
    .rename('r')
    .sort_values(
        key=abs,
        ascending=False
    )
)

pairs.head(10)

Attack   Total      0.754465
Sp. Atk  Total      0.746982
Sp. Def  Total      0.691391
HP       Total      0.669066
Defense  Total      0.598596
Speed    Total      0.561656
Defense  Sp. Def    0.528286
HP       Attack     0.518707
Sp. Atk  Sp. Def    0.480027
HP       Sp. Atk    0.456355
Name: r, dtype: float64

In [61]:
# SAVING 
pairs.head(20).to_csv(
    OUT / 'pokemon_top_correlations.csv'
)

*Your three findings here.*

# Findings
1. Total is most strongly correlated with the offensive and defensive base stats because it is computed as the sum of the six battle attributes.
2. non-Legendary Pokémon and account for many of the extreme outliers in the dataset.

3. Generations differ in both average strength and Legendary rate, showing that Pokémon distributions are not identical across generations.

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `06-OutputFiles`. Upload that folder along with this notebook.

In [62]:
for path in sorted(OUT.rglob('*')):
    print(path)

06-OutputFiles\bill_correlation_by_species.csv
06-OutputFiles\cleaned_penguins.csv
06-OutputFiles\correlation_matrix.csv
06-OutputFiles\measurement_summary.csv
06-OutputFiles\penguins_data_dictionary.csv
06-OutputFiles\pokemon_generation_summary.csv
06-OutputFiles\pokemon_measurement_summary.csv
06-OutputFiles\pokemon_top_correlations.csv
06-OutputFiles\sex_differences_by_species.csv
06-OutputFiles\species_summary.csv


---
### 📝 Summary
This lab described a dataset from scratch, with numbers only:
- **Inspected** first, removed only rows with nothing to contribute, and kept partial rows
- **Recorded meaning** with a data dictionary and `category` dtypes
- **Summarized** center, spread, and shape, then checked outliers overall **and within groups**
- **Compared species** fairly, and spotted island as a stand-in for species
- **Tested a relationship within groups** and found Simpson's paradox

🎓 That's Module 07. **Module 08** picks up right here: the same questions, answered with charts. Its first notebook starts with the Anscombe datasets from notebook 05, and its lab returns to these penguins.